# C1 · 2. Features: acoustic and XLS-R

For every clip from notebook 1:
- **acoustic** features (MFCC, pitch, energy, repetition and prolongation cues): `app/audio/prosody.py`, the same function the service runs;
- **XLS-R** (`facebook/wav2vec2-xls-r-300m`, multilingual, frozen): the clip's mean-pooled hidden state at several layers, so notebook 3 can pick the best layer on the dev split.

Both are cached in chunks under `DATA/features/`, so the run resumes. On an Apple-silicon Mac the encoder runs on the GPU (`mps`); expect a few hours for all of SEP-28k, and about 1.2 GB for the model download on first use.

Settings: `SWARA_C1_ENCODER`, `SWARA_C1_LAYERS` (default `6,9,12,15,18,21,24`), `SWARA_C1_DEVICE` (`mps`, `cuda` or `cpu`), `SWARA_C1_BATCH` (default 16), `SWARA_C1_WORKERS` (acoustic, default 4).

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the speech service's own code (app/, training/)
DATA = Path(os.environ.get("SWARA_C1_DATA", ROOT / "data" / "sep28k"))

ENCODER = os.environ.get("SWARA_C1_ENCODER", "facebook/wav2vec2-xls-r-300m")
LAYERS = [int(x) for x in os.environ.get("SWARA_C1_LAYERS", "6,9,12,15,18,21,24").split(",")]
DEVICE = os.environ.get("SWARA_C1_DEVICE") or None
BATCH = int(os.environ.get("SWARA_C1_BATCH", 16))
WORKERS = int(os.environ.get("SWARA_C1_WORKERS", 4))
from training import features
index = pd.read_csv(DATA / "clips_index.csv", dtype={"EpId": str, "ClipId": str})
paths = index["path"].tolist()
print(f"{len(paths)} clips; encoder {ENCODER}, layers {LAYERS}, device {DEVICE or features.best_device()}")

In [ ]:
t0 = time.time()
A = features.acoustic(paths, DATA / "features", workers=WORKERS)
print("acoustic:", A.shape, f"{time.time() - t0:.0f} s")

In [ ]:
t0 = time.time()
X = features.xlsr(paths, DATA / "features", ENCODER, LAYERS, device=DEVICE, batch=BATCH)
print({layer: x.shape for layer, x in X.items()}, f"{time.time() - t0:.0f} s")

In [ ]:
# One file per feature set, aligned row by row with clips_index.csv.
np.save(DATA / "features" / "acoustic.npy", A)
for layer, x in X.items():
    np.save(DATA / "features" / f"xlsr_L{layer}.npy", x)
(DATA / "features" / "meta.json").write_text(json.dumps({"encoder": ENCODER, "layers": LAYERS, "clips": len(paths)}))
print("saved to", DATA / "features")